In [ ]:
import os
import pandas as pd
from numpy import random
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
wd = os.getcwd()

In [ ]:
tcgaPurity = pd.read_excel('https://static-content.springer.com/esm/art%3A10.1038%2Fncomms9971/MediaObjects/41467_2015_BFncomms9971_MOESM1236_ESM.xlsx',skiprows=3)
tcgaPurity

In [ ]:
for i in ['50','60','70','80']:
    tmp = pd.read_csv('./data/403-puritymean'+i+'-a10/403.a10.purity.csv', index_col=0)
    tmp['Cancer type'] = i
    tmp['CPE'] = tmp['0']
    tcgaPurity = pd.concat([tcgaPurity, tmp])

tmp = pd.read_csv('./data/403-LGGGBM-a10/403.a10.purity.csv', index_col=0)
tmp['Cancer type'] = 'LGGGBM'
tmp['CPE'] = tmp['0']
tcgaPurity = pd.concat([tcgaPurity, tmp])

In [ ]:
tcgaPurity.dropna(subset=['CPE'])['Cancer type'].value_counts().loc[
    tcgaPurity.groupby('Cancer type')['CPE'].mean().sort_values().index
]

In [ ]:
import matplotlib.pyplot as plt
fig,ax = plt.subplots(figsize=[15,5])
sns.boxplot(ax=ax,x=tcgaPurity['Cancer type'], \
               y=tcgaPurity['CPE'],\
               order=tcgaPurity.groupby('Cancer type')['CPE'].mean().sort_values().index, color='white')
plt.savefig('./figures/ED3f.pdf')

In [ ]:
tcgaPurity[['Cancer type','CPE']].to_csv('../SourceData/Fig.ED3f.txt',sep='\t')
tcgaPurity[['Cancer type','CPE']]

In [ ]:
def getmetdmr(ab, abab):
    data = pd.read_table("./data/"+ab+"-fg-a10/beta_"+abab+"_a10.tsv")
    cpgs = data[['chr','pos']]
    cpgs['end'] = cpgs['pos']
    
    DMRs = pd.read_table("./data/"+ab+"-fg-a10/DMRs_"+abab+"_a10.bed", header=None).sort_values([1,2])
    DMRs.index = DMRs[0]+'.'+DMRs[1].astype(str)+'.'+DMRs[2].astype(str)
    DMRs['type'] = DMRs[6].apply(lambda x:('0,1,2|3,4L' if x[:4]=='leaf' else x))
    DMRs[6] = DMRs[6]+(DMRs[3]>0).map({True:'(P)',False:'(N)'}) +'@'+(DMRs[7]).astype(str)
    DMRs['leaf'] = DMRs[6].apply(lambda x:(x.split('|')[1] if x[:4]=='leaf' else None))
    DMRs['c'] = DMRs[7]
    DMRs['len'] = DMRs[2]-DMRs[1]
    DMRs['mdiff'] = DMRs[3]
    
    from pybedtools import BedTool
    def find_overlapping_regions_df(bed1_df, bed2_df, bed1_cols, bed2_cols):
        bed_1 = BedTool.from_dataframe(bed1_df[bed1_cols].sort_values(bed1_cols))
        bed_2 = BedTool.from_dataframe(bed2_df[bed2_cols].sort_values(bed2_cols))
        return BedTool.to_dataframe(bed_1.intersect(bed_2, wa=True, wb=True))
    
    dmrcpgs = find_overlapping_regions_df(DMRs, cpgs, \
                                            [0,1,2,6,8], ['chr','pos','end'])
    dmrcpgs.index = dmrcpgs['thickStart']
    
    metdmr = data.loc[~data['pos'].map(dmrcpgs['name'].to_dict()).isna()]
    for i in range(50):
        ctr = pd.read_table('./bg/'+ab+'/sample_'+str(i)+'.txt', header=None, index_col=1)
        metdmr['5_s'+str(i)] = metdmr['pos'].map(ctr[4]/ctr[5])
        data['5_s'+str(i)] = data['pos'].map(ctr[4]/ctr[5])
    metdmr['name'] = metdmr['pos'].map(dmrcpgs['name'].to_dict())
    metdmr['name'] = metdmr['name'].apply(lambda x:('0,1,2|3,4L('+x.split('@')[0].split('(')[1] if x.find('leaf')>-1 else x.split('@')[0]))
    metdmr['name'] = metdmr['name'] + '#' + metdmr['pos'].map(dmrcpgs['score'].to_dict()).astype(str)
    return metdmr, DMRs, data

In [ ]:
ab = '155'
abab = '15_5_10_10'
metdmrlow, DMRlows, datalow = getmetdmr(ab, abab)

ab = '403'
abab = '40_3_22_22'
metdmr, DMRs, data = getmetdmr(ab, abab)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
f,a = plt.subplots(5,2,figsize=[6,7], sharey=True,gridspec_kw={'width_ratios': [2, 1]})

data_grp = data[['chr','pos']]
for i in range(6):
    data_grp['mean_G'+str(i+1)] = data[data.columns[data.columns.str.contains(str(i)+'_s')]].T.mean()

sd_dmrdist = []

for ii,i in enumerate(['0|others(N)','0,1,2|3,4(N)','0,1|2(N)','3|4(N)','0,1,2|3,4L(N)']):
    tmp0 = pd.melt(metdmr.loc[metdmr['name']==i+'#0'].drop(columns=['chr','pos','name']))
    tmp0['grp'] = tmp0['variable'].apply(lambda x:'G'+str(1+int(x.split('_')[0])))
    tmp0['bg'] = '403'
    tmp01 = pd.melt(metdmrlow.loc[metdmrlow['name']==i+'#0'].drop(columns=['chr','pos','name']))
    tmp01['grp'] = tmp0['variable'].apply(lambda x:'G'+str(1+int(x.split('_')[0])))
    tmp01['bg'] = '155'
    tmp0 = pd.concat([tmp0, tmp01]).sort_values(['grp','bg'], ascending=[1,0])
    tmp0['type'] = i
    sd_dmrdist.append(tmp0)
    violincdict = {}
    for j,grp in enumerate(sorted(tmp0['grp'].unique())):
        violincdict[grp+'155'] = sns.color_palette("Set2")[j]
        violincdict[grp+'403'] = tuple([i*0.8 for i in violincdict[grp+'155']])
    sns.violinplot(x=tmp0['grp'], hue=tmp0['bg'], y=tmp0['value'], \
                   # palette=violincdict, \
                   split=True, gap=10, inner=None, ax=a[ii][0])

    collections = a[ii][0].collections

    for j,grp in enumerate(sorted(tmp0['grp'].unique())):
        collections[2*j].set_facecolor(violincdict[grp+'403'])
        collections[2*j+1].set_facecolor(violincdict[grp+'155'])
    
    a[ii][0].spines['top'].set_visible(False)
    a[ii][0].spines['right'].set_visible(False)
    a[ii][0].set_xlabel(None)
    a[ii][0].set_ylabel(None)
    a[ii][0].legend_.remove()
    
    if 1:#ii<4:
        a[ii][0].spines['bottom'].set_visible(False)
        a[ii][0].xaxis.set_visible(False)

    if i!='0,1,2|3,4L(N)':
        pos_ab = DMRs.loc[(DMRs[6].apply(lambda x:x.split('@')[0])==i)&(DMRs[3]<0)&\
                (DMRs[7]==0.87)].sort_values('len',ascending=False).iloc[0]
    else:
        pos_ab = DMRs.loc[(DMRs[6].apply(lambda x:x.split('@')[0])=='leaf|12(N)')&(DMRs[3]<0)&\
                (DMRs[7]==0.87)].sort_values('len',ascending=False).iloc[0]

    flen = 750
    for j in sorted(tmp0['grp'].unique()):
        tmp1 = data_grp.loc[(data_grp['pos']>=pos_ab[1]-flen)\
                &(data_grp['pos']<=pos_ab[2]+flen)]
        tmp2 = tmp1[:-1]
        tmp2['pos'] = list(tmp1['pos'][1:]-0.01)
        tmp1 = pd.concat([tmp1, tmp2])
        
        sns.lineplot(x=list(tmp1['pos']),\
                     y=list(tmp1['mean_'+j]),\
                     ax=a[ii][1], linestyle='-', estimator=None, linewidth=1, color=sns.color_palette("Set2")[int(j[1:])-1])

    a[ii][1].spines['top'].set_visible(False)
    a[ii][1].spines['right'].set_visible(False)
    a[ii][1].spines['bottom'].set_visible(False)
    a[ii][1].set_ylabel(None)
    a[ii][1].set_yticks([0,0.5,1])
    a[ii][1].xaxis.set_visible(False)

    a[ii][0].set_ylim([-0.1,1.1])
    a[ii][1].set_ylim([-0.1,1.1])

plt.savefig('./figures/2a.pdf',bbox_inches='tight')

In [ ]:
pd.concat(sd_dmrdist).to_csv('../SourceData/Fig.2a.txt',sep='\t')
pd.concat(sd_dmrdist)

In [ ]:
bg = '403'
sinfo = pd.read_csv('./data/'+bg+'-LGGGBM-a10/'+bg+'.a10.purity.csv',index_col=0)
sinfo['c1'] = sinfo.index.map(pd.read_csv('./data/'+bg+'-LGGGBM-a10/'+bg+'.a10.c1.csv',index_col=0)['0'])
sinfo['c2'] = sinfo.index.map(pd.read_csv('./data/'+bg+'-LGGGBM-a10/'+bg+'.a10.c2.csv',index_col=0)['0'])
sinfo['grp'] = [i.split('_')[0] for i in sinfo.index]
sinfo['batch'] = [int(i.split('s')[-1])%2 for i in sinfo.index]
sinfo.to_csv('../SourceData/Fig.ED2abc.txt',sep='\t')
sinfo

In [ ]:
plt.subplots(figsize=[3,3])
sns.histplot(sinfo['0'], bins=10, element="step")
plt.xlim([0.55,1.05])
plt.xticks([0.6,0.8,1])
plt.yticks([0,5,10])
plt.savefig('./figures/ED2a.pdf',bbox_inches='tight')
sinfo['0'].mean(), sinfo['0'].std()

In [ ]:
plt.subplots(figsize=[3,3])
sns.histplot(sinfo['c1'], bins=10, element="step")
plt.xlim([-0.05,1.05])
plt.xticks([0,0.5,1])
plt.yticks([0,5,10])
plt.savefig('./figures/ED2b.pdf',bbox_inches='tight')

In [ ]:
plt.subplots(figsize=[3,3])
sns.histplot(sinfo['c2'], bins=10, element="step")
plt.xlim([0.25,0.75])
plt.xticks([0.3,0.5,0.7])
plt.yticks([0,5,10])
plt.savefig('./figures/ED2c.pdf',bbox_inches='tight')

In [ ]:
dmrmet = pd.read_table('./data/403-LGGGBM-a10/403a10.metilene3.tsv')
# DMRs.loc[(DMRs[6]=='nogrp(N)@1.0') & DMRs.index.str.replace('.','-').isin(set(dmrmet.index))]
dmrmet = dmrmet.loc[(dmrmet['chr']=='chr10')&(dmrmet['pos']>=103279489)&(dmrmet['pos']<=103280946)].drop(columns=['chr','pos'])
plt.subplots(figsize=[3,3])
sns.regplot(x=sinfo['c1'],y=dmrmet.mean())
plt.xticks([0,0.5,1])
plt.yticks([0,0.5,1])
plt.xlim([-0.05,1.05])
plt.ylim([-0.05,1.05])
plt.savefig('./figures/ED2d.pdf',bbox_inches='tight')

In [ ]:
pd.DataFrame(dmrmet.mean()).to_csv('../SourceData/Fig.ED2d.txt', sep='\t')
pd.DataFrame(dmrmet.mean())